# OOD Detection Results

The following demonstrates how to reproduce the OOD detection experiments on TUAB.

In [ ]:
from oddeeg.utils import submit_eval_jobs, submit_train_jobs

## Pre-process Data

It is recommended to first pre-process the data using multiple cpus for parallelisation.

```bash
uv run oddeeg-preprocess --dataset_name TUAB --num_worker 32
```

## Training

Submit slurm jobs:

In [ ]:
dsc_train_cfg = {
    "dataset_name": "TUAB",
    "task": "normality",
    "model_name": "TCN",
    "training_mode": "discriminative",
}

gen_train_cfg = {
    "dataset_name": "TUAB",
    "task": None,  # unconditional generative model
    "model_name": "UNet",
    "training_mode": "flow_matching",
    "max_batches": 1000000,
}

Note: use `help(submit_train_jobs)` for info on supported options to specify slurm partition, gpu type, etc.

In [ ]:
submit_train_jobs([dsc_train_cfg, gen_train_cfg])

Alternatively, via CLI:

**Discriminative Model (TCN)**
```bash
uv run oddeeg-train \
    --dataset_name TUAB \
    --task normality \
    --model_name TCN \
    --training_mode discriminative
```

**Generative Model (Unet)**
```bash
uv run oddeeg-train \
    --dataset_name TUAB \
    --task None \
    --model_name UNet \
    --training_mode flow_matching \
    --max_batches 1000000
```


## Evaluation

The following jobs should be submitted once training is done (requires the trained models).

Submit slurm jobs:

In [ ]:
PERTURBATION_SWEEP = {
    "perturbation_sfreq": [125, 150, 200],
    "perturbation_channel_shuffle": [0.1, 0.5, 1.0],
    "perturbation_highpass_hz": [2.0, 4.0, 8.0],
    "perturbation_lowpass_hz": [45.0, 30.0, 15.0],
    "perturbation_reref_scheme": ["cz", "linked_temporal", "bipolar"],
}

eval_cfgs = []
for train_cfg in [dsc_train_cfg, gen_train_cfg]:
    for perturbation_key, strengths in PERTURBATION_SWEEP.items():
        for strength in strengths:
            eval_cfgs.append(
                {
                    "config": train_cfg,
                    perturbation_key: strength,
                }
            )

In [ ]:
submit_eval_jobs(eval_cfgs)

Alternatively, via CLI (example for sampling rate perturbation to 125 Hz):

```bash
uv run oddeeg-eval \
    --config path/to/your/training_output/config.yml \
    --perturbation_sfreq 125
```

Note that evaluations for the unperturbed (in-distribution) data are automatically executed with the train runs.

## Analysis

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score

from oddeeg.utils import construct_results_path
from oddeeg.aggregators import KDE, MaxQuantile

In [ ]:
METRICS = {
    "msp": {"label": "MSP", "higher_is_ood": False},
    "energy": {"label": "energy", "higher_is_ood": True},
    "odin": {"label": "ODIN", "higher_is_ood": False},
    "ash": {"label": "ASH", "higher_is_ood": True},
    "log_likelihood": {"label": "Log Likelihood", "higher_is_ood": False},
    "typicality": {"label": "Typicality", "higher_is_ood": True},
    "dose": {"label": "DoSE", "higher_is_ood": False},
    "sitn": {"label": "SITN", "higher_is_ood": True},
}

PERTURBATION_META = {
    "perturbation_sfreq": {"id_sentinel": 100},
    "perturbation_channel_shuffle": {"id_sentinel": 0},
    "perturbation_highpass_hz": {"id_sentinel": 0.0},
    "perturbation_lowpass_hz": {"id_sentinel": float("inf")},
    "perturbation_reref_scheme": {"id_sentinel": "average"},
}

In [ ]:
# Fit OOD detection methods using in-distribution validation data
preds_id_val = pd.read_csv(construct_results_path(config=gen_train_cfg, split_pick="val"))

if "typicality" in METRICS:
    entropy_estimate = np.mean(preds_id_val["log_likelihood"])
if "dose" in METRICS:
    dose = KDE(features=["log_likelihood", "source_log_likelihood", "log_determinant"])
    dose.fit(preds_id_val, subsample=10000)
if "sitn" in METRICS:
    sitn = MaxQuantile({"anderson_darling_statistic": True, "ps_cv": True})
    sitn.fit(preds_id_val)


In [ ]:
# Load in-distribution test predictions
preds_id_test_gen = pd.read_csv(construct_results_path(config=gen_train_cfg, split_pick="test"))
preds_id_test_dsc = pd.read_csv(construct_results_path(config=dsc_train_cfg, split_pick="test"))
preds_id_test = preds_id_test_gen.merge(preds_id_test_dsc)

# Compute OOD detection performance for each perturbation and strength level
records = []
for pert_key, strengths in PERTURBATION_SWEEP.items():
    id_sentinel = PERTURBATION_META.get(pert_key, {}).get("id_sentinel", 0)

    for strength in strengths:
        # Load out-of-distribution test predictions
        results_path = construct_results_path(config=gen_train_cfg, split_pick="test", **{pert_key: strength})
        if not results_path.exists():
            print(f"[WARN] Results not found: {results_path}")
            continue
        preds_ood_gen = pd.read_csv(results_path)

        results_path = construct_results_path(config=dsc_train_cfg, split_pick="test", **{pert_key: strength})
        if not results_path.exists():
            print(f"[WARN] Results not found: {results_path}")
            continue
        preds_ood_dsc = pd.read_csv(results_path)

        preds_ood = preds_ood_gen.merge(preds_ood_dsc)
        preds_ood[pert_key] = strength

        preds_id = preds_id_test.copy()
        preds_id[pert_key] = id_sentinel

        preds = pd.concat([preds_id, preds_ood], ignore_index=True)

        # Add OOD scores with fitted methods
        if "typicality" in METRICS:
            preds["typicality"] = (preds["log_likelihood"] - entropy_estimate).abs()
        if "dose" in METRICS:
            preds["dose"] = dose.score(preds)
        if "sitn" in METRICS:
            preds["sitn"] = sitn.score(preds)

        # Binary OOD label: 1 = OOD, 0 = ID
        y_true = (preds[pert_key] != id_sentinel).astype(int)

        for col, m in METRICS.items():
            if col not in preds.columns:
                continue
            scores = preds[col].copy()
            if not m["higher_is_ood"]:
                scores = -scores
            try:
                auroc = roc_auc_score(y_true, scores)
            except Exception as e:
                auroc = float("nan")
                print(f"[WARN] AUROC failed for {pert_key}={strength}, metric={col}: {e}")

            records.append(
                {
                    "perturbation_type": pert_key,
                    "strength": strength,
                    "metric": m["label"],
                    "AUROC": auroc,
                }
            )

auroc_table = pd.DataFrame(records).set_index(["perturbation_type", "strength", "metric"])
auroc_table